# Hito 6 — Modelos de ensamble: Random Forest y XGBoost con busqueda de hiperparametros

**Proyecto 1 de Innovacion Tecnologica — Maestria en Inteligencia Artificial Aplicada (MIAA)**
**Universidad Icesi · Cali, Colombia**
**Responsables del Hito 6:** Carlos Alberto Salamando · Veronica Serna
**Equipo:** Juan Fernando Gutierrez · Miguel Angel Paz · Carlos Alberto Salamando · Veronica Serna

---

## Objetivo

Entrenar Random Forest y XGBoost sobre las 90 features acusticas del Hito 3, con busqueda de
hiperparametros **dentro** de la validacion cruzada (RandomizedSearchCV, 12 candidatos x
GroupKFold(3) interno por autor), y compararlos contra el mejor modelo del Hito 5 (SVM RBF,
Macro-F1 = 0,466) con intervalos de confianza del 95 % y prueba pareada de Wilcoxon.
Protocolo multietiqueta con agregacion por grabacion (maximo por especie) — metodologia completa en
[`docs/hito6_ensembles_report.md`](../docs/hito6_ensembles_report.md).

**No se usa la particion `test` en ningun momento de este hito.**

Este notebook **lee los artefactos ya ejecutados** en `experiments/results/hito6/` (la corrida
completa tarda horas incluso en GPU). Para reproducirla desde cero:

```bash
uv run python src/main.py --stage hito6
```

## Paso 1 — Imports, rutas y carga de artefactos

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS = PROJECT_ROOT / "experiments" / "results" / "hito6"
summary = json.loads((RESULTS / "cv_summary.json").read_text(encoding="utf-8"))
paired = json.loads((RESULTS / "paired_tests_vs_hito5.json").read_text(encoding="utf-8"))
thresholds = json.loads((RESULTS / "threshold_sensitivity.json").read_text(encoding="utf-8"))
cv_scores = pd.read_csv(RESULTS / "cv_scores_by_fold.csv")
importance = pd.read_csv(RESULTS / "feature_importance.csv")

cfg = summary["config"]
print(f"Semilla: {cfg['seed']} | folds: {cfg['folds']} | busqueda interna: {cfg['inner_cv']}")
print(f"Modelos: {cfg['models']} | dispositivo XGBoost: {cfg['xgboost_device']}")
print(f"Especies: {cfg['n_species']} | SMOTE: {cfg['smote']}")

Semilla: 42 | folds: ['fold0', 'fold1', 'fold2', 'fold3', 'fold4'] | busqueda interna: GroupKFold(3) por autor, solo dentro del fold de entrenamiento
Modelos: ['random_forest', 'xgboost'] | dispositivo XGBoost: cuda
Especies: 16 | SMOTE: no aplicado: recall macro Hito 5 = 0.60-0.88 (sin colapso); remuestreo multietiqueta queda como trabajo condicional


## Paso 2 — Resumen por modelo (media ± std e IC 95 % entre los 5 folds)

Los intervalos usan la t de Student con n = 5 folds: son orientativos (potencia baja), no concluyentes.

In [2]:
rows = []
for model, metrics in summary["summary_by_model"].items():
    row = {"modelo": model}
    for metric, stats in metrics.items():
        row[metric] = f"{stats['mean']:.3f} [{stats['ci95_low']:.3f}, {stats['ci95_high']:.3f}]"
    rows.append(row)
table = pd.DataFrame(rows).set_index("modelo")
print("Referencia Hito 5 — SVM RBF: Macro-F1 = 0,466")
print(table.to_string())

Referencia Hito 5 — SVM RBF: Macro-F1 = 0,466
                           accuracy              f1_macro       precision_macro          recall_macro         roc_auc_macro
modelo                                                                                                                     
random_forest  0.112 [0.099, 0.126]  0.200 [0.188, 0.211]  0.841 [0.776, 0.905]  0.121 [0.111, 0.131]  0.888 [0.883, 0.893]
xgboost        0.276 [0.186, 0.366]  0.536 [0.506, 0.567]  0.484 [0.397, 0.572]  0.651 [0.591, 0.712]  0.908 [0.905, 0.912]


## Paso 3 — Tabla completa por fold x modelo

In [3]:
pd.set_option("display.max_columns", None)
print(cv_scores.to_string(index=False))

 fold         model  n_train_clips  n_val_clips  n_val_recordings  cv_f1_macro_inner_best  accuracy  f1_macro  precision_macro  recall_macro  roc_auc_macro     cmap  seconds
fold0 random_forest          14458         3435              1158                0.098852  0.094991  0.190600         0.912500      0.112466       0.893686 0.567913    205.6
fold0       xgboost          14458         3435              1158                0.514247  0.203800  0.511386         0.424199      0.681146       0.907833 0.613589   1199.9
fold1 random_forest          14353         3540              1153                0.118314  0.121422  0.204983         0.841540      0.124528       0.889377 0.572074    247.2
fold1       xgboost          14353         3540              1153                0.522150  0.354727  0.567996         0.574225      0.607682       0.909072 0.629288   1216.2
fold2 random_forest          14235         3658              1163                0.105631  0.116079  0.195902         0.785446    

## Paso 4 — Comparacion pareada contra SVM RBF (Hito 5): Wilcoxon

Diferencia de Macro-F1 fold a fold contra el SVM RBF del Hito 5, con prueba de Wilcoxon
(con n = 5 folds el p-valor minimo alcanzable es 0,0625).

In [4]:
for model, res in paired.items():
    diffs = ", ".join(f"{d:+.3f}" for d in res["diff_f1_macro_per_fold"])
    print(f"{model}: diffs por fold = [{diffs}]")
    print(f"  media = {res['mean_diff']:+.3f} | Wilcoxon p = {res['wilcoxon_p']}")
    print(f"  {res['note']}")

random_forest: diffs por fold = [-0.253, -0.281, -0.289, -0.261, -0.246]
  media = -0.266 | Wilcoxon p = 0.0625
  n=5 folds: potencia baja; interpretar como evidencia orientativa, no concluyente.
xgboost: diffs por fold = [+0.068, +0.082, +0.052, +0.097, +0.055]
  media = +0.071 | Wilcoxon p = 0.0625
  n=5 folds: potencia baja; interpretar como evidencia orientativa, no concluyente.


## Paso 5 — Sensibilidad al umbral de decision

El umbral fijo 0,5 penaliza a Random Forest (precision alta, recall bajo). Ajustando un umbral
por especie sobre el propio fold de validacion (protocolo de la primera entrega, sin tocar test):

In [5]:
rows = []
for model, res in thresholds.items():
    rows.append({
        "modelo": model,
        "F1 umbral 0,5": summary["summary_by_model"][model]["f1_macro"]["mean"],
        "F1 umbral ajustado": f"{res['f1_macro_mean']:.3f} ± {res['f1_macro_std']:.3f}",
        "umbral medio": round(sum(f["threshold_mean"] for f in res["per_fold"]) / len(res["per_fold"]), 3),
    })
print(pd.DataFrame(rows).set_index("modelo").to_string())
print()
print(thresholds["random_forest"]["note"])

               F1 umbral 0,5 F1 umbral ajustado  umbral medio
modelo                                                       
random_forest         0.1998      0.546 ± 0.014         0.220
xgboost               0.5365      0.610 ± 0.009         0.661

Umbral por especie elegido maximizando F1 sobre el propio fold de validación (protocolo de la primera entrega). No usa test.


## Paso 6 — Umbrales out-of-fold (OOF)

El ajuste de umbral por especie sobre un solo fold de validacion (fold4) mostro sobreajuste
(caso extremo: rumfly1 con umbral 0,30 y 111 falsos positivos en la primera evaluacion de test).
La solucion adoptada — documentada en [`docs/decision_umbrales_oof.md`](../docs/decision_umbrales_oof.md) —
ajusta los umbrales sobre las predicciones out-of-fold de los 5 folds (17.893 clips, sin test).
Se reproduce con `uv run python scripts/hito6_oof_thresholds.py`.

In [6]:
oof = json.loads((RESULTS / "oof_thresholds.json").read_text(encoding="utf-8"))
print(oof["nota"])
print(f"Macro-F1 val con umbrales fold4: {oof['macro_f1_val_umbrales_fold4']:.3f}")
print(f"Macro-F1 val con umbrales OOF:   {oof['macro_f1_val_umbrales_oof']:.3f}")
rows = []
for sp in cfg["species"]:
    t4, to = oof["thresholds_fold4"][sp], oof["thresholds_oof"][sp]
    rows.append({"especie": sp, "umbral_fold4": t4, "umbral_oof": to, "cambio": round(to - t4, 2)})
print(pd.DataFrame(rows).set_index("especie").to_string())

Umbrales ajustados con predicciones out-of-fold de los 5 folds (17.893 clips; sin test).
Macro-F1 val con umbrales fold4: 0.570
Macro-F1 val con umbrales OOF:   0.593
         umbral_fold4  umbral_oof  cambio
especie                                  
bugtan           0.49        0.56    0.07
chbant1          0.54        0.72    0.18
colcha1          0.62        0.64    0.02
compau           0.74        0.77    0.03
grekis           0.64        0.77    0.13
gycwor1          0.61        0.64    0.03
laufal1          0.56        0.64    0.08
linwoo1          0.55        0.52   -0.03
littin1          0.69        0.72    0.03
orcpar           0.63        0.71    0.08
paltan1          0.51        0.51    0.00
roahaw           0.56        0.74    0.18
rumfly1          0.30        0.55    0.25
soulap1          0.61        0.83    0.22
trsowl           0.55        0.71    0.16
wbwwre1          0.70        0.73    0.03


## Paso 7 — Importancia de caracteristicas (top 15 por modelo)

In [7]:
for model in ("random_forest", "xgboost"):
    agg = (
        importance[importance["model"] == model]
        .groupby("feature")["importance"]
        .agg(importance_mean="mean", importance_std="std")
        .nlargest(15, "importance_mean")
        .reset_index()
    )
    print(f"--- {model} ---")
    print(agg.to_string(index=False))
    print()

--- random_forest ---
               feature  importance_mean  importance_std
     mfcc_06_base_mean         0.064606        0.026248
     mfcc_07_base_mean         0.063873        0.026763
     mfcc_02_base_mean         0.063479        0.012606
spectral_skewness_mean         0.038710        0.017338
     mfcc_12_base_mean         0.031562        0.005805
     mfcc_11_base_mean         0.027062        0.009127
     mfcc_05_base_mean         0.024445        0.006279
     mfcc_08_base_mean         0.019866        0.005575
     mfcc_04_base_mean         0.019829        0.002100
      mfcc_06_base_std         0.019815        0.010870
     mfcc_03_base_mean         0.018400        0.003277
     mfcc_12_delta_std         0.017914        0.008110
    mfcc_12_delta2_std         0.017362        0.007971
     mfcc_01_base_mean         0.016858        0.004650
    mfcc_04_delta2_std         0.016315        0.006114

--- xgboost ---
           feature  importance_mean  importance_std
  mfcc_01_bas

## Paso 8 — Figuras de importancia de caracteristicas

Generadas por la corrida oficial en `experiments/results/hito6/figures/`.

In [8]:
from IPython.display import Image, display
for png in sorted((RESULTS / "figures").glob("feature_importance_*.png")):
    print(png.name)
    display(Image(filename=str(png)))

feature_importance_random_forest.png
<IPython.core.display.Image object>
feature_importance_xgboost.png
<IPython.core.display.Image object>


## Paso 9 — Conclusiones

1. **XGBoost es el mejor modelo del proyecto**: Macro-F1 = 0,536 [IC95 0,506, 0,567]
   y ROC-AUC = 0,908, superando al SVM RBF del Hito 5 (0,466) en los 5 folds
   (Wilcoxon ΔF1 = +0,071, p = 0,0625 — minimo alcanzable con n = 5).
2. El F1 bajo de Random Forest con umbral 0,5 es un **artefacto del umbral**, no del ranking:
   su ROC-AUC es 0,888 y con umbral por especie pasa de 0,200 a 0,547. XGBoost sube a 0,610.
3. Los umbrales OOF (Paso 6) corrigen el sobreajuste del ajuste sobre un solo fold:
   Macro-F1 en validacion 0,570 → 0,593, y el umbral extremo de rumfly1 (0,30 → 0,55).
4. Las importancias convergen en MFCC (medias y desviaciones) y ZCR como rasgos dominantes.
5. Evaluacion final en test (21-nov, `--stage evaluacion_final`): **Macro-F1 = 0,592,
   ROC-AUC = 0,915** — coincide con la estimacion de validacion (0,593), sin brecha de
   generalizacion. Detalle en [`docs/decision_umbrales_oof.md`](../docs/decision_umbrales_oof.md).

Para forzar CPU (sin GPU NVIDIA): `uv run python src/main.py --stage hito6 --device cpu`.